In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Lê a tabela Bronze
df = spark.table("workspace.bronze.tb_movies_info")

# 1) Renomeia as colunas
df = (
    df.withColumnRenamed("id", "id_filme")
      .withColumnRenamed("title", "titulo")
      .withColumnRenamed("original_title", "titulo_original")
      .withColumnRenamed("release_date", "data_lancamento_raw")
      .withColumnRenamed("runtime", "duracao_minutos")
      .withColumnRenamed("original_language", "idioma_original")
      .withColumnRenamed("status", "status_raw")
      .withColumnRenamed("overview", "sinopse")
      .withColumnRenamed("tagline", "frase_divulgacao")
)

# 2) Normaliza o status ANTES de traduzir
status_normalizado = F.upper(F.trim(F.regexp_replace(F.col("status_raw"), r"[-_]+", " ")))
status_normalizado = F.trim(F.regexp_replace(status_normalizado, r"\s+", " "))
df = df.withColumn("status_normalizado", status_normalizado)

# 3) Traduz o status normalizado.
df = df.withColumn(
    "status_filme",
    F.when(F.col("status_normalizado") == "RELEASED", "Lançado")
     .when(F.col("status_normalizado") == "POST PRODUCTION", "Pós-Produção")
     .when(F.col("status_normalizado") == "IN PRODUCTION", "Em Produção")
     .when(F.col("status_normalizado") == "PLANNED", "Planejado")
     .when(F.col("status_normalizado") == "RUMORED", "Rumores")
     .when(F.col("status_normalizado").isin("CANCELED", "CANCELLED"), "Cancelado")
     .otherwise("Não Informado")
)

# 4) Tratamento de data multi-formato: tenta vários padrões em ordem;
#    coalesce pega o primeiro que der certo. try_to_date tolera formato errado (retorna NULL em vez de erro).
df = df.withColumn(
    "data_lancamento",
    F.coalesce(
        F.try_to_date("data_lancamento_raw", F.lit("yyyy-MM-dd")),
        F.try_to_date("data_lancamento_raw", F.lit("dd/MM/yyyy")),
        F.try_to_date("data_lancamento_raw", F.lit("MM/dd/yyyy")),
        F.try_to_date("data_lancamento_raw", F.lit("dd-MM-yyyy")),
        F.try_to_date("data_lancamento_raw", F.lit("MM-dd-yyyy")),
    )
)

# 5) Coluna derivada: ano de lançamento
df = df.withColumn("ano_lancamento", F.year("data_lancamento"))

# 6) Deduplicação: mantém só o registro mais recente por filme, com base na ingestion_datetime
janela = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(janela)).filter("rn = 1").drop("rn")

# 7) Seleciona só as colunas finais, na ordem certa
df_silver_info = df.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao"
)

df_silver_info.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_info_filmes")

display(df_silver_info)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_cambio = spark.table("workspace.bronze.tb_cotacao_dolar")

# Extrai só a data (sem hora) e renomeia
df_cambio = (
    df_cambio.withColumn("data_cotacao", F.to_date("dataHoraCotacao"))
             .withColumnRenamed("cotacaoCompra", "valor_dolar")
             .select("data_cotacao", "valor_dolar")
             .dropDuplicates(["data_cotacao"])
)

# Descobre o intervalo de datas disponível e gera um calendário contínuo (todos os dias, sem buracos)
data_min, data_max = df_cambio.selectExpr("min(data_cotacao)", "max(data_cotacao)").first()

df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)) AS data_cotacao
""")

# Junta o calendário completo com as cotações que existem (fins de semana ficam com valor_dolar = NULL)
df_completo = df_calendario.join(df_cambio, "data_cotacao", "left")

# Forward Fill: para cada data, pega o último valor não-nulo visto até ali (olhando pra trás)
janela_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
df_silver_cambio = df_completo.withColumn(
    "valor_dolar",
    F.last("valor_dolar", ignorenulls=True).over(janela_ffill)
)

df_silver_cambio.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_cotacao_dolar")

display(df_silver_cambio)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.tb_movies_financials")

df = df.withColumnRenamed("id", "id_filme")

# Deduplicação: mantém sempre o registro mais recente por id_filme
window_dedup = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(window_dedup)).filter(F.col("rn") == 1).drop("rn")

taxa_dolar = (
    spark.table("workspace.silver.tb_cotacao_dolar")
    .orderBy(F.col("data_cotacao").desc())
    .select("valor_dolar")
    .first()["valor_dolar"]
)
print(f"Taxa de conversão USD->BRL utilizada: {taxa_dolar}")

def limpar_e_converter(col_raw):
    # Trata textos que representam ausência de dado como NULL, antes de qualquer conversão
    tratado = F.when(
        F.trim(F.upper(col_raw)).isin("UNKNOWN", "NÃO INFORMADO", "N/A", "NA", "NULL", ""),
        None
    ).otherwise(col_raw)

    tratado_upper = F.trim(F.upper(tratado))

    # Detecta abreviação K (mil) / M (milhão) / B (bilhão) no final do valor, ex: "34.0M" -> multiplicador 1.000.000
    multiplicador = (
        F.when(tratado_upper.rlike("[0-9](K)$"), F.lit(1000))
         .when(tratado_upper.rlike("[0-9](M)$"), F.lit(1000000))
         .when(tratado_upper.rlike("[0-9](B)$"), F.lit(1000000000))
         .otherwise(F.lit(1))
    )

    # Remove símbolos de moeda, espaços, letras (K/M/B/USD) e mantém só dígitos, ponto e sinal negativo
    limpo = F.regexp_replace(tratado, r"[^0-9.\-]", "")
    limpo = F.when(limpo == "", None).otherwise(limpo)

    # Faz o cast do número "base" (sem o multiplicador ainda) e só depois aplica o fator K/M/B
    valor_base = limpo.try_cast("decimal(18,4)")
    valor = F.round(valor_base * multiplicador, 2).try_cast("decimal(18,2)")

    # Regra de negócio: valores zerados ou negativos são tratados como ausentes
    return F.when(valor <= 0, None).otherwise(valor)

df = (
    df.withColumn("orcamento_usd", limpar_e_converter(F.col("budget")))
      .withColumn("receita_usd", limpar_e_converter(F.col("revenue")))
)

df = (
    df.withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(taxa_dolar), 2))
      .withColumn("receita_brl", F.round(F.col("receita_usd") * F.lit(taxa_dolar), 2))
)

df = (
    df.withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))
      .withColumn("lucro_brl", F.col("receita_brl") - F.col("orcamento_brl"))
      .withColumn(
          "margem_lucro_percentual",
          F.round(F.try_divide(F.col("lucro_usd"), F.col("receita_usd")) * 100, 2)
      )
)

df_silver_financeiro = df.select(
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual"
)

df_silver_financeiro.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_financeiro_filmes")

display(df_silver_financeiro)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.tb_movies_metrics")

df = df.withColumnRenamed("id", "id_filme")

window_dedup = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(window_dedup)).filter(F.col("rn") == 1).drop("rn")

# Camada 1: se o valor bruto contém letra, é texto vazado (sinopse/tag), não é número -> NULL
popularidade_bruta = F.col("popularity")
eh_texto = popularidade_bruta.rlike("(?i)[a-zà-ÿ]")

popularidade_limpa = F.regexp_replace(popularidade_bruta, ",", ".")
popularidade_limpa = F.regexp_replace(popularidade_limpa, r"[^0-9.\-]", "")
df = df.withColumn(
    "popularidade",
    F.when(eh_texto, None).otherwise(popularidade_limpa.try_cast("double"))
)

# Camada 2: número inteiro "redondo" dentro da faixa de anos de lançamento plausíveis (1870-2030)
# tem muito mais cara de ano vazado (Column Shift) do que de popularidade real, que quase sempre
# vem com casas decimais -> NULL
eh_ano_vazado = (F.col("popularidade") == F.floor(F.col("popularidade"))) & (F.col("popularidade").between(1870, 2030))
df = df.withColumn("popularidade", F.when(eh_ano_vazado, None).otherwise(F.col("popularidade")))

df = (
    df.withColumn("nota_media_tmdb", F.col("vote_average").try_cast("double"))
      .withColumn("qtd_votos_tmdb", F.col("vote_count").try_cast("int"))
      .withColumn("nota_media_imdb", F.col("averageRating").try_cast("double"))
      .withColumn("qtd_votos_imdb", F.col("numVotes").try_cast("int"))
)

# Camada 3: limites de negócio (notas 0-10, contagens/popularidade não-negativas, teto de sanidade)
df = (
    df.withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
      .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
      .withColumn("popularidade", F.when(F.col("popularidade").between(0, 10000), F.col("popularidade")))
      .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
      .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
)

df_silver_metricas = df.select(
    "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
)

df_silver_metricas.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_metricas_engajamento")

display(df_silver_metricas)

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.tb_movies_reviews")

df = (
    df.withColumnRenamed("id", "id_filme")
      .withColumnRenamed("nome", "nome_usuario")
      .withColumnRenamed("nota", "nota_usuario_raw")
      .withColumnRenamed("comentario", "comentario_usuario_raw")
)

# Remove duplicatas exatas: mesma combinação de filme, usuário, nota e comentário
df = df.dropDuplicates(["id_filme", "nome_usuario", "nota_usuario_raw", "comentario_usuario_raw"])

# Nota do usuário: conversão segura + regra de negócio (escala válida: 0 a 10)
df = df.withColumn("nota_usuario", F.col("nota_usuario_raw").try_cast("double"))
df = df.withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))

# Comentários vazios ou só com espaços em branco recebem texto padronizado
df = df.withColumn(
    "comentario_usuario",
    F.when(
        F.trim(F.coalesce(F.col("comentario_usuario_raw"), F.lit(""))) == "",
        F.lit("Sem comentário")
    ).otherwise(F.col("comentario_usuario_raw"))
)

df_silver_avaliacoes = df.select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")

df_silver_avaliacoes.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_avaliacoes_usuarios")

display(df_silver_avaliacoes)

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.tb_credits_and_tags")
df = df.withColumnRenamed("id", "id_filme")

# A origem usa "|" como separador principal, mas padronizamos vírgula/ponto-e-vírgula para "|" também, por segurança
generos_normalizado = F.regexp_replace(F.col("genres"), "[,;]", "|")

df_generos = df.withColumn("genero_raw", F.explode(F.split(generos_normalizado, r"\|")))

# Remove espaços e aspas sobressalentes que aparecem no dado sujo
df_generos = df_generos.withColumn(
    "nome_genero",
    F.trim(F.regexp_replace(F.col("genero_raw"), '"', ''))
)

# Lista fechada de gêneros válidos (taxonomia padrão TMDB).
# Qualquer valor fora dessa lista é resíduo de Column Shift (nomes, taglines, países, hashtags etc.) e é descartado.
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]

df_generos = df_generos.filter(F.col("nome_genero").isin(generos_validos))

df_silver_generos = df_generos.select("id_filme", "nome_genero").dropDuplicates()

df_silver_generos.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_generos")

display(df_silver_generos.select("nome_genero").distinct().orderBy("nome_genero"))

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.tb_credits_and_tags")
df = df.withColumnRenamed("id", "id_filme")

def extrair_entidades(df_origem, coluna_origem, tipo_entidade):
    normalizado = F.regexp_replace(F.col(coluna_origem), "[,;]", "|")

    exploded = df_origem.select(
        "id_filme",
        F.explode(F.split(normalizado, r"\|")).alias("nome_raw")
    )

    exploded = exploded.withColumn(
        "nome_pessoa_empresa",
        F.initcap(F.trim(F.regexp_replace(F.col("nome_raw"), '"', '')))
    )

    # Remove resíduos: vazios, puramente numéricos, textos longos demais,
    # e caminhos de arquivo/imagem (ex: "/abc123.jpg") que vazaram por Column Shift
    exploded = exploded.filter(
        (F.col("nome_pessoa_empresa") != "") &
        (F.length(F.col("nome_pessoa_empresa")) <= 50) &
        (~F.col("nome_pessoa_empresa").rlike(r"^[0-9.\-]+$")) &
        (~F.col("nome_raw").contains("/")) &
        (~F.lower(F.col("nome_raw")).rlike(r"\.(jpg|jpeg|png|gif)$"))
    )

    return exploded.withColumn("tipo_entidade", F.lit(tipo_entidade))

# Aplica a mesma função para os 4 tipos, conforme o mapeamento do enunciado
df_atores = extrair_entidades(df, "cast", "Ator")
df_diretores = extrair_entidades(df, "directors", "Diretor")
df_roteiristas = extrair_entidades(df, "writers", "Roteirista")
df_produtoras = extrair_entidades(df, "production_companies", "Produtora")

# Une os 4 num único modelo (dimensão unificada, como pede o enunciado)
df_pessoas_empresas = df_atores.unionByName(df_diretores).unionByName(df_roteiristas).unionByName(df_produtoras)

df_silver_pessoas_empresas = (
    df_pessoas_empresas
    .select("id_filme", "nome_pessoa_empresa", "tipo_entidade")
    .dropDuplicates(["id_filme", "nome_pessoa_empresa", "tipo_entidade"])
)

df_silver_pessoas_empresas.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_pessoas_empresas")

display(df_silver_pessoas_empresas)